### Query NewsAPI for multiple keywords and append results (topics | count | prevalence) from NewsAPI's indexed corpus

As an example, we will try these topics (with significant keywords) :

[
    "EV",
    "gas",
    "nuclear",
    "renewable",
    "climate"
]

In [ ]:
import requests
import pandas as pd

API_KEY = "6ebfd5fbe709457c8aca7a519699ed98"

queries = {
    "ev":
        '("electric vehicle" OR EV)',

    "gas":
        '("gasoline vehicle" OR petrol OR "carbone emission")',

    "nuclear":
        '("nuclear power" OR "nuclear energy")',

    "renewables":
        '("renewable energy" OR "clean energy" OR wind OR solar)',

    "climate":
        '("climate change" OR "global warming")'
} # these can be changed for significant keywords and subjects depending the researcher's goals

results = []

for topic, query in queries.items():

    r = requests.get(
        "https://newsapi.org/v2/everything",
        params={
            "q": query,
            "language": "en",
            "from" : "2026-06-15", # kept under a month back because of the free-tier date restriction
            "to"   : "2026-06-24", # change to suit the researcher's needs
            "sortBy": "publishedAt",
            "pageSize": 1,
            "apiKey": API_KEY
        }
    ).json()

    results.append({
        "topic": topic,
        "count": r["totalResults"]
    })

df = pd.DataFrame(results)
df["share"] = df["count"] / df["count"].sum()
print(df)

        topic  count     share
0          ev   1994  0.113631
1         gas    900  0.051288
2     nuclear    600  0.034192
3  renewables   7287  0.415261
4     climate   6767  0.385628


### If needed, we can also query for examples of recent articles containing some of our keywords and topics.
We can check the type of articles that come out recently, qualitatively compared to those we have in our database.

In [8]:
import requests
import pandas as pd
from datetime import datetime, timedelta

API_KEY = "6ebfd5fbe709457c8aca7a519699ed98"

base_url = "https://newsapi.org/v2/everything"

queries = {
    "renewables":
        '("renewable energy" OR "clean energy" OR wind OR solar)'
}

all_articles = []

for topic, query in queries.items():

    params = {
        "q": query,
        "language": "en",
        "sortBy": "publishedAt",
        "from": (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d"), # kept under a month back because of the free-tier date restriction
        "pageSize": 100,
        "apiKey": API_KEY
    }

    response = requests.get(base_url, params=params)
    data = response.json()

    for article in data.get("articles", []):
        all_articles.append({
            "topic": topic,
            "title": article["title"],
            "description": article["description"],
            "source": article["source"]["name"],
            "publishedAt": article["publishedAt"],
            "url": article["url"]
        })

df = pd.DataFrame(all_articles)

print(df.shape)
df.head()

(99, 6)


,topic,title,description,source,publishedAt,url
0,renewables,Texas lassoes massive Microsoft datacenter - a...,The air turns brown when bit barns come to tow...,Theregister.com,2026-06-22T20:31:10Z,https://www.theregister.com/on-prem/2026/06/22...
1,renewables,"For decades, we did not know whether Earth-lik...",For most of the time astronomers have thought ...,Space Daily,2026-06-22T20:30:14Z,https://spacedaily.com/t-for-decades-we-did-no...
2,renewables,Cleanup efforts underway after diesel spill in...,Nearly three years after a chemical spill in M...,CBC News,2026-06-22T20:29:05Z,https://www.cbc.ca/news/canada/toronto/diesel-...
3,renewables,Music mogul Clive Davis dies at 94,"During his six-decade career, Davis signed and...",Salon,2026-06-22T20:23:30Z,https://www.salon.com/2026/06/22/music-mogul-c...
4,renewables,Daystar Power Reaches 6.8 Megawatts of Install...,"LAGOS, Nigeria, June 22, 2026 /PRNewswire/ -- ...",PRNewswire,2026-06-22T20:23:00Z,https://www.prnewswire.com/news-releases/dayst...


# Conclusion

In this notebook, we:

- Tried to query NewsAPI for keyword frequency in recent news articles publications
- Tried to query for examples of such articles

We are now ready to look for keywords and specific topics in our datasets. If they are very prevalent in recent publications and mostly absent of our datasets, we can conclude that the models trained on our datasets will have performance issues generalizing to the new subjects in recent unseen data.